In [1]:

!pip install -q "sentence-transformers==2.7.0" "huggingface-hub<0.26.0" chromadb langchain langgraph langchain-community

!pip install -q beautifulsoup4 requests pymupdf python-docx
!pip install -q google-generativeai



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.5 MB

In [ ]:

import os

if not os.getenv('GEMINI_API_KEY') and os.getenv('GOOGLE_API_KEY'):
    os.environ['GEMINI_API_KEY'] = os.getenv('GOOGLE_API_KEY')

assert os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY'), (
    'Set GEMINI_API_KEY (or GOOGLE_API_KEY) in your environment before running.'
)


In [ ]:

import os
os.environ.setdefault('REBUILD_DB', 'false')
print('REBUILD_DB =', os.environ['REBUILD_DB'])


In [ ]:


import os
import json
import re
from typing import List, Dict, Any, TypedDict
from datetime import datetime
from urllib.parse import quote_plus, urlparse, parse_qs, unquote
import warnings
warnings.filterwarnings('ignore')

import requests
from bs4 import BeautifulSoup

try:
    import chromadb
    from chromadb.config import Settings
except ImportError as e:
    print(f"⚠️  ChromaDB import issue: {e}")
    print("Try: !pip install -q chromadb==0.4.24")

try:
    from sentence_transformers import SentenceTransformer
except ImportError as e:
    print(f"⚠️  Sentence-transformers import issue: {e}")
    print('Try: !pip install -q "sentence-transformers==2.7.0" "huggingface-hub<0.26.0"')

try:
    import google.generativeai as genai
except ImportError as e:
    print(f"⚠️  Google AI import issue: {e}")
    print("Try: !pip install -q google-generativeai")

try:
    from langgraph.graph import StateGraph, END
    from langchain.text_splitter import RecursiveCharacterTextSplitter
except ImportError as e:
    print(f"⚠️  LangGraph import issue: {e}")
    print("Try: !pip install -q langgraph langchain")


class Config:
    """System configuration"""

    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

    DATA_DIR = "/kaggle/working/legal_data"
    CHROMA_DIR = "/kaggle/working/chroma_db"

    EMBEDDING_MODEL = "all-MiniLM-L6-v2"
    LLM_MODEL = "models/gemini-2.0-flash"

    CHUNK_SIZE = 800
    CHUNK_OVERLAP = 200

    TOP_K = 5

   
    REBUILD_DB = os.getenv("REBUILD_DB", "false").lower() == "true"


os.makedirs(Config.DATA_DIR, exist_ok=True)
os.makedirs(Config.CHROMA_DIR, exist_ok=True)


class LegalDataIngester:
    """Handles fetching and processing legal documents"""

    def __init__(self):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=Config.CHUNK_SIZE,
            chunk_overlap=Config.CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " ", ""]
        )

    def fetch_us_code_sample(self) -> List[Dict[str, Any]]:
        """Fetch sample US Code data from Cornell LII"""
        print("Fetching US Code samples...")
        documents: List[Dict[str, Any]] = []

        sample_urls = [
            "https://www.law.cornell.edu/uscode/text/18/1001",  # False statements
            "https://www.law.cornell.edu/uscode/text/15/1",     # Sherman Act
            "https://www.law.cornell.edu/uscode/text/42/1983",  # Civil Rights
        ]

        for url in sample_urls:
            try:
                resp = requests.get(url, timeout=10)
                if resp.status_code != 200:
                    print(f"✗ HTTP {resp.status_code} for {url}")
                    continue

                soup = BeautifulSoup(resp.content, "html.parser")

                
                title_elem = soup.find("h1")
                title = title_elem.get_text(strip=True) if title_elem else "Unknown"

                
                content = ""
                candidates = [
                    soup.find("div", {"id": "tab-source-content"}),
                    soup.find("div", {"id": "content"}),
                    soup.find("main"),
                    soup.find("article"),
                ]
                for cand in candidates:
                    if cand:
                        content = cand.get_text(separator="\n", strip=True)
                        if content:
                            break

                if not content:
                    content = soup.get_text(separator="\n", strip=True)

               
                parts = url.rstrip("/").split("/")
                if len(parts) >= 2:
                    title_num = parts[-2]
                    section_num = parts[-1]
                    citation = f"{title_num} U.S.C. § {section_num}"
                else:
                    citation = url

                documents.append(
                    {
                        "content": content,
                        "metadata": {
                            "source": "US Code",
                            "citation": citation,
                            "title": title,
                            "url": url,
                            "jurisdiction": "Federal",
                            "type": "Statute",
                            "date_accessed": datetime.now().isoformat(),
                        },
                    }
                )
                print(f"✓ Fetched: {citation}")
            except Exception as e:
                print(f"✗ Error fetching {url}: {e}")

        return documents

    def fetch_federal_register_updates(self, days_back: int = 7) -> List[Dict[str, Any]]:
        """Fetch recent Federal Register updates"""
        print(f"Fetching Federal Register updates (last {days_back} days)...")
        documents: List[Dict[str, Any]] = []

        try:
            url = (
                "https://www.federalregister.gov/api/v1/documents.json"
                "?per_page=20&order=newest"
            )
            resp = requests.get(url, timeout=10)
            if resp.status_code != 200:
                print(f"✗ HTTP {resp.status_code} from Federal Register API")
                return documents

            data = resp.json()
            for doc in data.get("results", [])[:10]:
                content = f"{doc.get('title', '')}\n\n{doc.get('abstract', '')}"
                documents.append(
                    {
                        "content": content,
                        "metadata": {
                            "source": "Federal Register",
                            "title": doc.get("title", ""),
                            "document_number": doc.get("document_number", ""),
                            "publication_date": doc.get("publication_date", ""),
                            "url": doc.get("html_url", ""),
                            "jurisdiction": "Federal",
                            "type": "Regulation Update",
                        },
                    }
                )
            print(f"✓ Fetched {len(documents)} Federal Register updates")
        except Exception as e:
            print(f"✗ Error fetching Federal Register: {e}")

        return documents

    def fetch_supreme_court_opinions(self) -> List[Dict[str, Any]]:
        """Fetch recent Supreme Court opinions via CourtListener"""
        print("Fetching Supreme Court opinions...")
        documents: List[Dict[str, Any]] = []

        try:
            url = "https://www.courtlistener.com/api/rest/v3/opinions/?court=scotus&order_by=-date_filed"
            headers = {"User-Agent": "Legal-RAG-System/1.0"}
            resp = requests.get(url, headers=headers, timeout=10)
            if resp.status_code != 200:
                print(f"✗ HTTP {resp.status_code} from CourtListener")
                return documents

            data = resp.json()
            for opinion in data.get("results", [])[:5]:
                opinion_url = opinion.get("resource_uri", "")
                if not opinion_url:
                    continue

                full_url = f"https://www.courtlistener.com{opinion_url}"
                op_resp = requests.get(full_url, headers=headers, timeout=10)
                if op_resp.status_code != 200:
                    continue

                op_data = op_resp.json()
                text = op_data.get("plain_text") or op_data.get("html", "") or ""
                documents.append(
                    {
                        "content": text[:5000],
                        "metadata": {
                            "source": "Supreme Court Opinion",
                            "case_name": opinion.get("case_name", ""),
                            "citation": opinion.get("case_name", ""),
                            "date_filed": opinion.get("date_filed", ""),
                            "url": f"https://www.courtlistener.com{opinion.get('absolute_url', '')}",
                            "jurisdiction": "Federal",
                            "court": "SCOTUS",
                            "type": "Case",
                        },
                    }
                )
            print(f"✓ Fetched {len(documents)} Supreme Court opinions")
        except Exception as e:
            print(f"✗ Error fetching Supreme Court opinions: {e}")

        return documents

    def chunk_documents(self, documents: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Split documents into chunks"""
        print("Chunking documents...")
        chunked_docs: List[Dict[str, Any]] = []

        for doc in documents:
            chunks = self.text_splitter.split_text(doc["content"])
            for i, chunk in enumerate(chunks):
                meta = dict(doc["metadata"])
                meta.update(
                    {
                        "chunk_id": i,
                        "total_chunks": len(chunks),
                    }
                )
                chunked_docs.append(
                    {
                        "content": chunk,
                        "metadata": meta,
                    }
                )

        print(f"✓ Created {len(chunked_docs)} chunks from {len(documents)} documents")
        return chunked_docs


class LegalVectorDB:
    """Manages ChromaDB vector database"""

    def __init__(self):
        self.embedding_model = SentenceTransformer(Config.EMBEDDING_MODEL)
        self.client = chromadb.PersistentClient(path=Config.CHROMA_DIR)
        self.collection = self.client.get_or_create_collection(
            name="us_law_knowledge_base",
            metadata={"description": "US Law documents with embeddings"},
        )
        print(f"✓ Vector DB initialized. Current documents: {self.collection.count()}")

    def add_documents(self, documents: List[Dict[str, Any]]):
        """Add documents to vector database"""
        if not documents:
            print("No documents to add")
            return

        print(f"Adding {len(documents)} chunks to vector database...")

        texts = [doc["content"] for doc in documents]
        embeddings = self.embedding_model.encode(texts, show_progress_bar=True)

        ids = [f"doc_{i}_{hash(doc['content'])}" for i, doc in enumerate(documents)]
        metadatas = [doc["metadata"] for doc in documents]

        batch_size = 100
        for i in range(0, len(documents), batch_size):
            end_idx = min(i + batch_size, len(documents))
            self.collection.add(
                embeddings=embeddings[i:end_idx].tolist(),
                documents=texts[i:end_idx],
                metadatas=metadatas[i:end_idx],
                ids=ids[i:end_idx],
            )

        print(f"✓ Added {len(documents)} documents. Total: {self.collection.count()}")

    def search(
        self,
        query: str,
        n_results: int = Config.TOP_K,
        filters: Dict[str, Any] | None = None,
    ) -> List[Dict[str, Any]]:
        """Search vector database"""
        query_embedding = self.embedding_model.encode([query])[0]

        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results,
            where=filters,
        )

        formatted_results: List[Dict[str, Any]] = []
        if results.get("documents"):
            for i in range(len(results["documents"][0])):
                formatted_results.append(
                    {
                        "content": results["documents"][0][i],
                        "metadata": results["metadatas"][0][i],
                        "distance": results["distances"][0][i],
                    }
                )

        return formatted_results


class AgentState(TypedDict):
    """State shared across agents"""

    query: str
    legal_domain: str
    jurisdiction: str
    retrieved_docs: List[Dict[str, Any]]
    verified_docs: List[Dict[str, Any]]
    final_answer: str
    citations: List[str]
    suggestions: List[str]


class LegalAgentSystem:
    """Agentic RAG system for legal queries"""

    def __init__(self, vector_db: LegalVectorDB):
        self.vector_db = vector_db

       
        try:
            genai.configure(api_key=Config.GEMINI_API_KEY)

            model_options = [
                "models/gemini-2.0-flash",
                "models/gemini-2.0-flash-lite",
            ]

            self.llm = None
            for model_name in model_options:
                try:
                    candidate = genai.GenerativeModel(model_name)
                    _ = candidate.generate_content("test")
                    self.llm = candidate
                    print(f"✓ Successfully initialized Gemini model: {model_name}")
                    Config.LLM_MODEL = model_name
                    break
                except Exception as e:
                    print(f"⚠️  Model {model_name} failed: {str(e)[:120]}")
                    continue
            else:
                raise RuntimeError("Could not initialize any Gemini 2.0 model")

        except Exception as e:
            print(f" Gemini initialization failed: {e}")
            print(" Tip: Make sure GEMINI_API_KEY / GOOGLE_API_KEY is set correctly.")
            raise

        self.workflow = self._build_workflow()


   
    def simple_web_search(self, question: str, max_results: int = 3) -> List[Dict[str, str]]:
        """
        Very simple web search using DuckDuckGo HTML results.
        Returns a list of {title, url}.
        """
        print("🌐 Web fallback: performing simple web search...")
        query = quote_plus(question + " law site:.gov OR site:.edu OR site:.org")
        url = f"https://duckduckgo.com/html/?q={query}"
        headers = {"User-Agent": "Legal-RAG-WebFallback/1.0"}

        try:
            resp = requests.get(url, headers=headers, timeout=10)

           
            if resp.status_code >= 400:
                print(f"✗ Web search HTTP {resp.status_code}")
                return []

            soup = BeautifulSoup(resp.text, "html.parser")
            results: List[Dict[str, str]] = []

            
            links = soup.select("a.result__a")

            
            if not links:
                links = soup.select("a[href]")

            for a in links:
                href = a.get("href")
                title = a.get_text(" ", strip=True)
                if not href or not title:
                    continue

               
                if href.startswith("#"):
                    continue

                results.append({"title": title, "url": href})

                if len(results) >= max_results:
                    break

            print(f"🌐 Web search got {len(results)} results")
            return results

        except Exception as e:
            print(f"✗ Error during web search: {e}")
            return []



 
    def fetch_web_pages(self, entries: List[Dict[str, str]]) -> List[Dict[str, str]]:
        """
        Fetch and extract text from a small list of web pages.
        Handles DuckDuckGo redirect URLs and protocol-relative URLs.
        Returns list of {title, url, content}.
        """
        def normalize_url(raw_url: str) -> str:
            url = raw_url

            
            if url.startswith("//"):
                url = "https:" + url

            
            if url.startswith("/"):
                url = "https://duckduckgo.com" + url

            
            try:
                parsed = urlparse(url)
                qs = parse_qs(parsed.query)
                if "uddg" in qs and qs["uddg"]:
                    return unquote(qs["uddg"][0])
            except Exception:
                
                pass

            return url

        pages: List[Dict[str, str]] = []
        headers = {"User-Agent": "Legal-RAG-WebFallback/1.0"}

        for entry in entries:
            raw_url = entry["url"]
            title = entry["title"]
            url = normalize_url(raw_url)

            try:
                resp = requests.get(url, headers=headers, timeout=10)
                if resp.status_code != 200:
                    print(f"✗ HTTP {resp.status_code} for {url}")
                    continue
                soup = BeautifulSoup(resp.text, "html.parser")
                text = soup.get_text(" ", strip=True)
                if not text:
                    continue
                pages.append(
                    {
                        "title": title,
                        "url": url,
                        "content": text[:8000],  
                    }
                )
            except Exception as e:
                print(f"✗ Error fetching {url}: {e}")
                continue

        print(f"🌐 Fetched {len(pages)} web pages for fallback context")
        return pages

    def answer_via_web(self, question: str) -> tuple[str, List[str]]:
        """
        Use simple web search + Gemini to answer when local DB has no relevant sources.
        Returns (answer_text, citations_list).
        """
        search_results = self.simple_web_search(question, max_results=3)
        if not search_results:
            msg = (
                "I couldn't find relevant primary sources in the local knowledge base, "
                "and web search returned no usable results. "
                "Please consult an up-to-date legal database or a licensed attorney."
            )
            return msg, []

        web_pages = self.fetch_web_pages(search_results)
        if not web_pages:
            msg = (
                "I attempted a web-based lookup but couldn't extract enough information "
                "to answer reliably. Please consult an up-to-date legal database or a licensed attorney."
            )
            return msg, []

        context = "\n\n---\n\n".join(
            [
                f"Title: {p['title']}\nURL: {p['url']}\nContent:\n{p['content']}"
                for p in web_pages
            ]
        )

        prompt = f"""You are a legal research assistant.

The local legal knowledge base had no relevant sources for this question,
so I fetched a few web pages that appear related.

Question: {question}

Web Sources (snippets):
{context}

Instructions:
1. Base your answer ONLY on the information in these web snippets.
2. Provide:
   - A clear explanation in plain language
   - Proper legal citations or statute references if they appear (e.g., 'K.S.A. 58-2564')
   - Mention any URLs that seem authoritative (e.g., .gov or judiciary sites)
3. Be explicit about limitations or uncertainty.
4. If the snippets are clearly insufficient, say so.

Format your response with clear sections.
"""

        try:
            resp = self.llm.generate_content(prompt)
            answer_text = resp.text

            
            pattern = (
                r"\b\d+\s+U\.S\.C\.\s+§\s*\d+[A-Za-z0-9]*"
                r"|\bK\.S\.A\.\s*\d+-\d+[a-zA-Z0-9]*"
                r"|\b[A-Z][A-Za-z]+ v\. [A-Z][A-Za-z]+"
            )
            found = re.findall(pattern, answer_text)
            citations = list(dict.fromkeys(found))  

            return answer_text, citations
        except Exception as e:
            msg = f"Error generating web-based fallback answer: {e}"
            return msg, []

    

    def _build_workflow(self) -> StateGraph:
        """Build the agent workflow graph"""
        workflow = StateGraph(AgentState)

        workflow.add_node("router", self.router_agent)
        workflow.add_node("retriever", self.retrieval_agent)
        workflow.add_node("verifier", self.citation_agent)
        workflow.add_node("synthesizer", self.synthesis_agent)

        workflow.set_entry_point("router")
        workflow.add_edge("router", "retriever")
        workflow.add_edge("retriever", "verifier")
        workflow.add_edge("verifier", "synthesizer")
        workflow.add_edge("synthesizer", END)

        return workflow.compile()

    def router_agent(self, state: AgentState) -> AgentState:
        """Determines legal domain and jurisdiction"""
        query = state["query"]

        prompt = f"""Analyze this legal query and determine:
1. Legal domain (e.g., criminal, civil, constitutional, regulatory, antitrust, civil rights, landlord-tenant)
2. Jurisdiction (federal, state, or both)

Query: {query}

Respond in JSON format:
{{"legal_domain": "...", "jurisdiction": "..."}}"""

        try:
            resp = self.llm.generate_content(prompt)
            text = resp.text.strip().replace("```json", "").replace("```", "")
            result = json.loads(text)
            state["legal_domain"] = result.get("legal_domain", "general")
            state["jurisdiction"] = result.get("jurisdiction", "federal")
        except Exception:
            state["legal_domain"] = "general"
            state["jurisdiction"] = "federal"

        print(
            f"🎯 Router: Domain={state['legal_domain']}, "
            f"Jurisdiction={state['jurisdiction']}"
        )
        return state

    def retrieval_agent(self, state: AgentState) -> AgentState:
        """Retrieves relevant legal documents"""
        query = state["query"]
        jurisdiction = state["jurisdiction"]

        filters: Dict[str, Any] | None = None
        if jurisdiction.lower() != "both":
            filters = {"jurisdiction": {"$eq": jurisdiction.capitalize()}}

        results = self.vector_db.search(query, n_results=Config.TOP_K, filters=filters)
        state["retrieved_docs"] = results

        print(f"📚 Retriever: Found {len(results)} relevant documents")
        return state

    def citation_agent(self, state: AgentState) -> AgentState:
        """Verifies citations and currency of law"""
        docs = state["retrieved_docs"]
        verified: List[Dict[str, Any]] = []

        for doc in docs:
            meta = doc.get("metadata", {})
            # keep statutes, regs, and cases
            if (
                meta.get("citation")
                or meta.get("document_number")
                or meta.get("case_name")
            ):
                verified.append(doc)

        state["verified_docs"] = verified
        print(f"✅ Verifier: Verified {len(verified)} documents with proper citations")
        return state

    def synthesis_agent(self, state: AgentState) -> AgentState:
        """Generates final answer with citations"""
        query = state["query"]
        docs = state["verified_docs"]

        # Fallback: no verified local docs → use web search
        if not docs:
            print("💡 Synthesizer: No verified local docs, falling back to web search (requests-based)")
            answer, citations = self.answer_via_web(query)
            state["final_answer"] = answer
            state["citations"] = citations
            return state

        # Normal RAG path using local documents
        context = "\n\n---\n\n".join(
            [
                (
                    f"Source: {doc['metadata'].get('source', 'Unknown')}\n"
                    f"Citation or Title: "
                    f"{doc['metadata'].get('citation', doc['metadata'].get('title', 'N/A'))}\n"
                    f"Content: {doc['content'][:500]}..."
                )
                for doc in docs[:3]
            ]
        )

        prompt = f"""You are an expert legal assistant. Answer this legal question based on the provided sources.

Question: {query}

Legal Sources:
{context}

Provide:
1. A clear, accurate answer based on the sources
2. Proper legal citations (statutes, regulations, or case names as available)
3. Practical suggestions or next steps
4. Any warnings or limitations

If the sources do NOT actually address the question, explicitly say so and explain the mismatch.

Format your response with clear sections."""

        try:
            resp = self.llm.generate_content(prompt)
            state["final_answer"] = resp.text

            citations = [
                doc["metadata"].get(
                    "citation",
                    doc["metadata"].get("title", "Unknown"),
                )
                for doc in docs
            ]
            state["citations"] = citations
        except Exception as e:
            state["final_answer"] = f"Error generating response: {e}"
            state["citations"] = []

        print(
            f"💡 Synthesizer: Generated final answer with {len(state['citations'])} citations"
        )
        return state

    def query(self, question: str) -> Dict[str, Any]:
        """Process a legal query through the agent system"""
        initial_state: AgentState = {
            "query": question,
            "legal_domain": "",
            "jurisdiction": "",
            "retrieved_docs": [],
            "verified_docs": [],
            "final_answer": "",
            "citations": [],
            "suggestions": [],
        }

        final_state = self.workflow.invoke(initial_state)

        return {
            "question": question,
            "answer": final_state["final_answer"],
            "citations": final_state["citations"],
            "domain": final_state["legal_domain"],
            "jurisdiction": final_state["jurisdiction"],
        }



def main():
    """Main execution function"""

    print("=" * 70)
    print("US LAW AGENTIC RAG SYSTEM")
    print("=" * 70)

  
    print("\n[1/5] Initializing components...")
    ingester = LegalDataIngester()
    vector_db = LegalVectorDB()

   
    if Config.REBUILD_DB or vector_db.collection.count() == 0:
        print("\n[2/5] Fetching legal data...")
        documents: List[Dict[str, Any]] = []

        documents.extend(ingester.fetch_us_code_sample())
        documents.extend(ingester.fetch_federal_register_updates())
        documents.extend(ingester.fetch_supreme_court_opinions())

        chunked_docs = ingester.chunk_documents(documents)

        print("\n[3/5] Building vector database...")
        vector_db.add_documents(chunked_docs)
    else:
        print("\n[2/5] Using existing vector database")
        print("[3/5] Vector database ready")

    
    print("\n[4/5] Initializing agent system...")
    agent_system = LegalAgentSystem(vector_db)

    print("\n[5/5] Testing system with sample queries...")
    print("=" * 70)

    test_queries = [
        "What are the penalties for making false statements to federal agents?",
        "What is the Sherman Antitrust Act about?",
        "What are my rights under 42 USC 1983?",
        "What are the Kansas eviction notice requirements?",
    ]

    for q in test_queries:
        print("\n" + "=" * 70)
        print(f"QUERY: {q}")
        print("=" * 70)

        result = agent_system.query(q)

        print(f"\n📋 Domain: {result['domain']}")
        print(f"🏛️  Jurisdiction: {result['jurisdiction']}")
        print(f"\n💬 ANSWER:\n{result['answer']}")
        print("\n📚 CITATIONS:")
        for i, cit in enumerate(result["citations"], 1):
            print(f"  {i}. {cit}")
        print()

    print("=" * 70)
    print(" System ready! You can now use agent_system.query('your question')")
    print("=" * 70)

    return agent_system, vector_db


if __name__ == "__main__":
    agent_system, vector_db = main()

    print("\n💡 Try it yourself:")
    print("result = agent_system.query('Your legal question here')")
    print("print(result['answer'])")


US LAW AGENTIC RAG SYSTEM

[1/5] Initializing components...
✓ Vector DB initialized. Current documents: 160

[2/5] Fetching legal data...
Fetching US Code samples...
✓ Fetched: 18 U.S.C. § 1001
✓ Fetched: 15 U.S.C. § 1
✓ Fetched: 42 U.S.C. § 1983
Fetching Federal Register updates (last 7 days)...
✓ Fetched 10 Federal Register updates
Fetching Supreme Court opinions...
✗ HTTP 401 from CourtListener
Chunking documents...
✓ Created 160 chunks from 13 documents

[3/5] Building vector database...
Adding 160 chunks to vector database...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

✓ Added 160 documents. Total: 160

[4/5] Initializing agent system...
✓ Successfully initialized Gemini model: models/gemini-2.0-flash

[5/5] Testing system with sample queries...

QUERY: What are the penalties for making false statements to federal agents?
🎯 Router: Domain=Criminal, Jurisdiction=Federal
📚 Retriever: Found 5 relevant documents
✅ Verifier: Verified 5 documents with proper citations
💡 Synthesizer: Generated final answer with 5 citations

📋 Domain: Criminal
🏛️  Jurisdiction: Federal

💬 ANSWER:
### Answer to Question: Penalties for Making False Statements to Federal Agents

Based on the provided sources, specifically 18 U.S.C. § 1001, the penalties for making false statements to federal agents are as follows:

1.  **Fines:** The individual may be fined under Title 18. The specific amount is not detailed within the provided source.

2.  **Imprisonment:**
    *   Generally, imprisonment for not more than 5 years.
    *   If the offense involves international or domestic terr